# Performance of Linear Regression Model in Predicting Fuel Consumption for the Government Fleets
**Guimaras State University — College of Science and Technology**

This notebook reproduces the model-development workflow of the study (Chapter III) in Google Colab:

1. Merge the local trip records (Monthly Report of Official Travel) with the vehicle records by plate number,
   and add the supplementary dataset with the same variables
2. Data cleaning: drop missing values and zero distance; IQR outliers of fuel per 100 km within each engine-power group
3. Encoding (fuel type: 1 = Gasoline, 0 = Diesel) and the interaction term Distance × Engine Power
4. Multicollinearity check with the Variance Inflation Factor (VIF above 5 = high collinearity), with distance and
   engine power mean-centered before the interaction term is formed
5. Train-test split (80/20) and Linear Regression training:
   **ŷ = β0 + β1(Distance) + β2(Engine Power) + β3(Fuel Type) + β4(Distance × Engine Power)**
6. Evaluation with R², MAE and RMSE, and 5-fold cross-validation
7. Residual plot, residual histogram and Durbin-Watson test on the local trips arranged by date of travel
8. Saving the trained model with Joblib for the Flask web application

The code is the same preprocessing and diagnostics code used by the web application
(`model/preprocess.py`, `model/diagnostics.py`, `model/train_model.py`).

In [ ]:
# Colab already includes pandas, numpy, scikit-learn, matplotlib and seaborn.
!pip -q install statsmodels joblib

In [ ]:
import json, os, warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
sns.set_theme(style="whitegrid")
RANDOM_STATE, CV_FOLDS = 42, 5

## 1. Data collection — upload the datasets
Download the three files from the web application: **Model Performance → Training Details →**
**trip_records.csv**, **vehicle_records.csv** and **supplementary_data.csv**, then upload them when the file picker appears.

* **trip_records.csv** — the Monthly Report of Official Travel: date of travel, plate number, distance traveled (km)
  and fuel consumed (L) of every local trip.
* **vehicle_records.csv** — plate number, vehicle model, fuel type and engine power (kW).
* **supplementary_data.csv** — the supplementary dataset with the same variables.

In [ ]:
TRIPS_PATH, VEHICLES_PATH, SUPPLEMENTARY_PATH = "trip_records.csv", "vehicle_records.csv", "supplementary_data.csv"
try:
    from google.colab import files  # running in Google Colab
    if not all(os.path.exists(p) for p in (TRIPS_PATH, VEHICLES_PATH, SUPPLEMENTARY_PATH)):
        files.upload()
except ImportError:
    print("Not in Colab: put the three CSV files next to this notebook.")
print(TRIPS_PATH, VEHICLES_PATH, SUPPLEMENTARY_PATH)

## 2. Preprocessing and diagnostics code (the web application's `model/preprocess.py` and `model/diagnostics.py`)
The two files are written next to the notebook and imported, so the saved model can be loaded by the web application.

In [ ]:
os.makedirs("model", exist_ok=True)
open(os.path.join("model", "__init__.py"), "a").close()

In [ ]:
%%writefile model/preprocess.py
"""
Shared preprocessing for the fuel-consumption Linear Regression model
(Chapter III, How the Project Will Work). Imported by BOTH train_model.py and
app.py, so the exact same features and encoding are used when the model is
trained and when it serves a live prediction.

Predictors (Scope): distance traveled (km), engine power (kW) and fuel type;
the predicted value is the fuel consumed per trip in liters. The model is

    y^ = b0 + b1(Distance) + b2(Engine Power) + b3(Fuel Type) + b4(Distance x Engine Power)

with fuel type encoded as 1 = Gasoline, 0 = Diesel.
"""
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

FUEL_TYPES = ("Diesel", "Gasoline")
FUEL_TYPE_CODES = {"Diesel": 0, "Gasoline": 1}  # Encoding: 1 represents gasoline and 0 represents diesel

# The three inputs a prediction needs (from the trip and the selected vehicle's record).
FEATURE_COLUMNS = ["distance_km", "engine_power_kw", "fuel_type"]
# The four terms of the regression equation, after encoding and the interaction term.
MODEL_TERMS = ["distance_km", "engine_power_kw", "fuel_type_code", "distance_x_power"]
TERM_LABELS = {
    "distance_km": "Distance (km)",
    "engine_power_kw": "Engine Power (kW)",
    "fuel_type_code": "Fuel Type (1 = Gasoline)",
    "distance_x_power": "Distance × Engine Power",
    "actual_fuel_consumption": "Fuel Consumed (L)",
}
TARGET_COLUMN = "actual_fuel_consumption"

# Kept for code that still asks for them: every model term is numeric now.
NUMERIC_FEATURES = MODEL_TERMS
CATEGORICAL_FEATURES = []

# The local Monthly Report of Official Travel export (one row per trip) and the
# vehicle records it is merged with on the plate number.
TRIP_COLUMNS = ["trip_date", "plate_number", "distance_km", TARGET_COLUMN]
VEHICLE_COLUMNS = ["plate_number", "brand_model", "fuel_type", "engine_power_kw"]


def encode_features(frame, centers=None):
    """Encoding and interaction term: fuel type as 1 (Gasoline) / 0 (Diesel),
    plus Distance x Engine Power for the higher fuel use per kilometre of
    larger engines. Returns the four model terms in a fixed order.

    With `centers` (mean distance, mean engine power of the training set),
    distance and power are mean-centered before the interaction is formed.
    Centering removes the collinearity between a predictor and an interaction
    built from it (high VIF) without changing any prediction."""
    frame = pd.DataFrame(frame)
    distance = pd.to_numeric(frame["distance_km"], errors="coerce").astype(float)
    power = pd.to_numeric(frame["engine_power_kw"], errors="coerce").astype(float)
    fuel = frame["fuel_type"].map(lambda v: FUEL_TYPE_CODES.get(str(v).strip().title(), float("nan"))).astype(float)
    if centers is not None:
        distance, power = distance - centers[0], power - centers[1]
    return pd.DataFrame({
        "distance_km": distance,
        "engine_power_kw": power,
        "fuel_type_code": fuel,
        "distance_x_power": distance * power,
    }, index=frame.index)


class TermEncoder(BaseEstimator, TransformerMixin):
    """Pipeline step: learns the mean distance and mean engine power of the
    training data (fit) and builds the four centered terms (transform)."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.distance_mean_ = float(pd.to_numeric(X["distance_km"], errors="coerce").mean())
        self.power_mean_ = float(pd.to_numeric(X["engine_power_kw"], errors="coerce").mean())
        return self

    def transform(self, X):
        return encode_features(X, centers=(self.distance_mean_, self.power_mean_))


def build_feature_encoder():
    """The encoding step bundled into the saved joblib pipeline in front of the
    regression, so a prediction is always encoded exactly like the training data."""
    return TermEncoder()


def raw_equation(intercept, coefficients, centers):
    """The fitted centered equation written back in the original units:
    y = b0 + b1(D - d) + b2(P - p) + b3 F + b4(D - d)(P - p)
      = [b0 - b1 d - b2 p + b4 d p] + (b1 - b4 p) D + (b2 - b4 d) P + b3 F + b4 D P."""
    b1, b2, b3, b4 = (coefficients[t] for t in MODEL_TERMS)
    d, p = centers
    return (intercept - b1 * d - b2 * p + b4 * d * p,
            {"distance_km": b1 - b4 * p, "engine_power_kw": b2 - b4 * d, "fuel_type_code": b3, "distance_x_power": b4})


def merge_trips_and_vehicles(trips, vehicles):
    """The trip records and the vehicle records are stored separately, so they are
    merged using the plate number (each trip gets its vehicle's engine power and
    fuel type)."""
    vehicles = vehicles[["plate_number", "fuel_type", "engine_power_kw"]].drop_duplicates("plate_number")
    trips = trips.drop(columns=[c for c in ("fuel_type", "engine_power_kw") if c in trips.columns])
    return trips.merge(vehicles, on="plate_number", how="left")


def supplementary_info(supplementary_path):
    """Describe the supplementary dataset: online trip records with the same
    variables (distance, engine power, fuel type, fuel consumed)."""
    columns = list(pd.read_csv(supplementary_path, nrows=0).columns)
    missing = [c for c in FEATURE_COLUMNS + [TARGET_COLUMN] if c not in columns]
    return {"kind": "trips", "dropped_features": [], "missing_columns": missing, "columns": columns}


def load_and_merge_datasets(historical_path, supplementary_path, extra_df=None):
    """Combine the local trips (the historical CSV plus the trip records in the
    system, already merged with their vehicles) and the supplementary dataset on
    their shared variables. Each record is labeled with its source
    (`data_source`), so the local trips can be identified after merging."""
    frames = []
    historical = pd.read_csv(historical_path)
    if not historical.empty:
        frames.append(historical.assign(data_source="historical"))
    if extra_df is not None and not extra_df.empty:
        frames.append(extra_df.assign(data_source="system"))
    supplementary = pd.read_csv(supplementary_path)
    frames.append(supplementary.assign(data_source="supplementary"))
    df = pd.concat(frames, ignore_index=True)
    for col in FEATURE_COLUMNS + [TARGET_COLUMN, "trip_date", "plate_number"]:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def clean_data(df):
    """Remove records with missing values or zero distance, then extreme
    outliers with the IQR rule on the fuel used per 100 km, within each group
    of vehicles with the same engine power (large vehicles are compared only
    with vehicles of similar size)."""
    df = df.copy()
    for col in ("distance_km", "engine_power_kw", TARGET_COLUMN):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["fuel_type"] = df["fuel_type"].astype("string").str.strip().str.title()
    df = df.dropna(subset=["distance_km", "engine_power_kw", "fuel_type", TARGET_COLUMN])
    df = df[df["fuel_type"].isin(FUEL_TYPES)]
    df = df[(df["distance_km"] > 0) & (df["engine_power_kw"] > 0) & (df[TARGET_COLUMN] > 0)]

    # IQR = Q3 - Q1; lower bound = Q1 - 1.5(IQR); upper bound = Q3 + 1.5(IQR).
    # The spread never counts as less than 25% of the group's median: a group
    # whose records all share one value (e.g. fuel written down as distance / 10)
    # would otherwise have an IQR of 0 and reject every real trip added later.
    def within_iqr(per_100km):
        q1, q3 = per_100km.quantile(0.25), per_100km.quantile(0.75)
        iqr = max(q3 - q1, 0.25 * per_100km.median())
        return per_100km.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

    per_100km = df[TARGET_COLUMN] / df["distance_km"] * 100
    keep = per_100km.groupby(df["engine_power_kw"], group_keys=False).apply(within_iqr)
    df = df[keep.reindex(df.index).fillna(False).astype(bool)]
    return df.reset_index(drop=True)


def build_prediction_frame(values):
    """A single-row frame from the trip distance and the selected vehicle's
    engine power and fuel type, in the schema the pipeline expects."""
    return pd.DataFrame([{
        "distance_km": float(values["distance_km"]),
        "engine_power_kw": float(values["engine_power_kw"]),
        "fuel_type": str(values["fuel_type"]),
    }])

In [ ]:
%%writefile model/diagnostics.py
"""
Regression diagnostics (Chapter II, Theoretical Background; Chapter III, How
the Project Will Work), computed with statsmodels:

* Variance Inflation Factor (VIF) of each predictor - a value above 5
  indicates high collinearity (reported; the equation's four terms are kept).
* LINE conditions on the residuals of an OLS fit of the same equation:
    Linearity        - Ramsey RESET test
    Independence     - Durbin-Watson statistic on the residuals of the LOCAL
                       trips arranged by the date of travel
    Normality        - Jarque-Bera test (with the residual histogram)
    Equal variance   - Breusch-Pagan test
* Coefficient standard errors and p-values use heteroskedasticity-robust
  (HC3) standard errors, which stay valid when the equal-variance condition
  is not met (larger errors on longer trips).
"""
import math
import warnings

import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson, jarque_bera

VIF_THRESHOLD = 5.0
ALPHA = 0.05
VIF_CAP = 1000.0  # perfect collinearity gives an infinite VIF; stored as ">1000"


def compute_vif(frame):
    """VIF_j = 1 / (1 - R_j^2) for every column of `frame`, with an intercept.
    A column with the same value in every record has no VIF (None)."""
    usable = [c for c in frame.columns if frame[c].astype(float).std() > 0]
    out = {c: None for c in frame.columns if c not in usable}
    if len(usable) < 2:
        out.update({c: 1.0 for c in usable})
        return out
    X = sm.add_constant(frame[usable].astype(float), has_constant="add")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # divide-by-zero for perfectly collinear columns
        for i, col in enumerate(X.columns):
            if col == "const":
                continue
            value = variance_inflation_factor(X.values, i)
            out[col] = VIF_CAP if (not math.isfinite(value) or value > VIF_CAP) else round(float(value), 2)
    return out


def _p(value):
    return None if value is None or not math.isfinite(float(value)) else round(float(value), 4)


def regression_diagnostics(terms_train, y_train, vif, local_residuals_by_date=None):
    """Fit OLS on the training split with the model's terms and test the LINE
    conditions. `local_residuals_by_date` are the residuals of the local trips
    in date order (for the Durbin-Watson test); without local trips the test
    falls back to the training residuals."""
    usable = [c for c in terms_train.columns if terms_train[c].astype(float).std() > 0]
    design = sm.add_constant(terms_train[usable].astype(float), has_constant="add")
    y = np.asarray(y_train, dtype=float)
    result = sm.OLS(y, design).fit()
    resid = np.asarray(result.resid)

    reset = linear_reset(result, power=2, use_f=True)
    if local_residuals_by_date is not None and len(local_residuals_by_date) >= 3:
        dw = float(durbin_watson(np.asarray(local_residuals_by_date, dtype=float)))
        dw_note = f"Residuals of the {len(local_residuals_by_date)} local trips arranged by date of travel."
    else:
        dw = float(durbin_watson(resid))
        dw_note = "No dated local trips yet: computed on the training residuals."
    jb_stat, jb_p, skew, kurtosis = jarque_bera(resid)
    bp_lm, bp_p, _, _ = het_breuschpagan(resid, design)
    vif_values = [v for v in vif.values() if v is not None]
    max_vif = max(vif_values) if vif_values else 0.0

    tests = [
        {"key": "linearity", "assumption": "Linearity", "test": "Ramsey RESET test",
         "statistic": round(float(reset.fvalue), 3), "p_value": _p(reset.pvalue),
         "passed": bool(reset.pvalue > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": "Fuel consumed is a straight-line function of the predictors."},
        {"key": "independence", "assumption": "Independence of errors", "test": "Durbin-Watson statistic",
         "statistic": round(dw, 3), "p_value": None,
         "passed": bool(1.5 <= dw <= 2.5), "rule": "1.5 to 2.5 (close to 2 = independent)",
         "meaning": dw_note},
        {"key": "normality", "assumption": "Normality of errors", "test": "Jarque-Bera test",
         "statistic": round(float(jb_stat), 3), "p_value": _p(jb_p),
         "passed": bool(jb_p > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": f"Residuals follow a normal distribution (skew {float(skew):.2f}, kurtosis {float(kurtosis):.2f})."},
        {"key": "equal_variance", "assumption": "Equal variance of errors", "test": "Breusch-Pagan test",
         "statistic": round(float(bp_lm), 3), "p_value": _p(bp_p),
         "passed": bool(bp_p > ALPHA), "rule": f"p > {ALPHA}",
         "meaning": "The spread of errors is the same for short and long trips."},
        {"key": "multicollinearity", "assumption": "No multicollinearity", "test": "Variance Inflation Factor",
         "statistic": round(float(max_vif), 2), "p_value": None,
         "passed": bool(max_vif <= VIF_THRESHOLD), "rule": f"every VIF ≤ {VIF_THRESHOLD:g}",
         "meaning": "Predictors do not carry the same information (an interaction term usually shares "
                    "information with the predictors it is built from)."},
    ]
    # Same coefficients; HC3 robust standard errors for the significance tests.
    robust = sm.OLS(y, design).fit(cov_type="HC3")
    coefficients = []
    for name in ["const"] + usable:
        coefficients.append({
            "feature": "Intercept" if name == "const" else name,
            "coef": float(f"{float(robust.params[name]):.4g}"),
            "std_err": float(f"{float(robust.bse[name]):.4g}"),
            "p_value": _p(robust.pvalues[name]),
            "significant": bool(robust.pvalues[name] < ALPHA),
        })
    return {
        "tests": tests,
        "coefficients": coefficients,
        "constant_terms": [c for c in terms_train.columns if c not in usable],
        "ols_r2": round(float(result.rsquared), 4),
        "ols_adj_r2": round(float(result.rsquared_adj), 4),
        "alpha": ALPHA,
        "std_err_type": "HC3 (heteroskedasticity-robust)",
    }

In [ ]:
from model.preprocess import (FEATURE_COLUMNS, MODEL_TERMS, TARGET_COLUMN, TERM_LABELS, build_feature_encoder,
                              clean_data, merge_trips_and_vehicles, raw_equation)
from model.diagnostics import VIF_THRESHOLD, compute_vif, regression_diagnostics

## 3. Merge and clean the data
The trips are merged with the vehicle records using the plate number; each record is labeled with its source.

In [ ]:
trips = pd.read_csv(TRIPS_PATH)
vehicles = pd.read_csv(VEHICLES_PATH)
local = merge_trips_and_vehicles(trips, vehicles).assign(data_source="system")
supplementary = pd.read_csv(SUPPLEMENTARY_PATH).assign(data_source="supplementary")
raw = pd.concat([f for f in (local, supplementary) if not f.empty], ignore_index=True)
df = clean_data(raw)
print(f"Records before cleaning: {len(raw)}  after cleaning: {len(df)}")
print(df["data_source"].value_counts().to_string())
df.head()

## 4. Train-test split (80/20), encoding, interaction term and multicollinearity (VIF)
Fuel type is encoded 1 = Gasoline, 0 = Diesel. Distance and engine power are mean-centered on the training set before the interaction term is formed (this removes the collinearity of the interaction without changing any prediction). A VIF above 5 indicates high collinearity.

In [ ]:
X, y = df[FEATURE_COLUMNS], df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Training set: {len(X_train)}  Testing set: {len(X_test)}")
encoder = build_feature_encoder().fit(X_train)
centers = (encoder.distance_mean_, encoder.power_mean_)
print(f"Mean distance {centers[0]:.4f} km, mean engine power {centers[1]:.4f} kW")
terms = encoder.transform(X)
vif = compute_vif(terms)
display(pd.DataFrame({"Term": [TERM_LABELS[t] for t in vif], "VIF": list(vif.values()),
                      "High collinearity (VIF > 5)": [v is not None and v > VIF_THRESHOLD for v in vif.values()]}))
plt.figure(figsize=(8, 6))
cols = [c for c in MODEL_TERMS if terms[c].nunique() > 1]
corr = terms[cols].assign(**{TARGET_COLUMN: y.values}).corr().rename(index=TERM_LABELS, columns=TERM_LABELS)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap of the Terms"); plt.show()

## 5. Train the Linear Regression model

In [ ]:
pipeline = Pipeline(steps=[("encoder", build_feature_encoder()), ("regressor", LinearRegression())])
pipeline.fit(X_train, y_train)
reg = pipeline.named_steps["regressor"]
symbols = {"distance_km": "Distance", "engine_power_kw": "Power", "fuel_type_code": "FuelType", "distance_x_power": "Distance×Power"}
coef = dict(zip(MODEL_TERMS, map(float, reg.coef_)))
D, P = f"(D - {centers[0]:.4f})", f"(P - {centers[1]:.4f})"
centered = {"distance_km": D, "engine_power_kw": P, "fuel_type_code": "(FuelType)", "distance_x_power": D + P}
print("Centered:       ŷ = " + f"{reg.intercept_:.6g}" + "".join(f" {'-' if c < 0 else '+'} {abs(c):.6g}{centered[t]}" for t, c in coef.items()))
raw_b0, raw_coef = raw_equation(float(reg.intercept_), coef, centers)
print("Original units: ŷ = " + f"{raw_b0:.6g}" + "".join(f" {'-' if c < 0 else '+'} {abs(c):.6g}({symbols[t]})" for t, c in raw_coef.items()))

## 6. Evaluate — R², MAE, RMSE and 5-fold cross-validation

In [ ]:
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
kfold = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv = cross_validate(pipeline, X, y, cv=kfold, scoring=("r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"))
cv_scores, cv_mae, cv_rmse = cv["test_r2"], -cv["test_neg_mean_absolute_error"], -cv["test_neg_root_mean_squared_error"]
display(pd.DataFrame({"Test set": [r2, mae, rmse],
                      "CV mean": [cv_scores.mean(), cv_mae.mean(), cv_rmse.mean()],
                      "CV std": [cv_scores.std(), cv_mae.std(), cv_rmse.std()]},
                     index=["R²", "MAE (L)", "RMSE (L)"]).round(4))

## 7. Regression assumption checks (LINE conditions)
Durbin-Watson is computed on the residuals of the local trips arranged by date of travel. Coefficient standard errors and p-values are heteroskedasticity-robust (HC3).

In [ ]:
local_rows = df[df["data_source"] == "system"]
local_resid = None
if not local_rows.empty:
    local_rows = local_rows.assign(_date=pd.to_datetime(local_rows["trip_date"], errors="coerce")).sort_values(["_date"], kind="stable")
    local_resid = (local_rows[TARGET_COLUMN] - pipeline.predict(local_rows[FEATURE_COLUMNS])).to_numpy()
diagnostics = regression_diagnostics(encoder.transform(X_train), y_train, vif, local_resid)
display(pd.DataFrame(diagnostics["tests"])[["assumption", "test", "statistic", "p_value", "rule", "passed"]])
pd.DataFrame(diagnostics["coefficients"])

## 8. Evaluation plots — actual vs predicted, residual plot and residual histogram

In [ ]:
residuals = y_test.values - y_pred
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(y_test, y_pred, alpha=0.4); lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax[0].plot(lim, lim, "--", color="teal"); ax[0].set(xlabel="Actual Fuel Consumed (L)", ylabel="Predicted (L)", title="Actual vs Predicted")
ax[1].scatter(y_pred, residuals, alpha=0.4); ax[1].axhline(0, ls="--", color="teal")
ax[1].set(xlabel="Predicted (L)", ylabel="Residual (Actual − Predicted)", title="Residual Plot")
sns.histplot(residuals, kde=True, ax=ax[2]); ax[2].axvline(0, ls="--", color="teal")
ax[2].set(xlabel="Residual (L)", ylabel="Number of Test Trips", title="Histogram of Residuals")
plt.tight_layout(); plt.show()

## 9. Save the trained model (Joblib) for the web application

In [ ]:
joblib.dump(pipeline, "linear_regression_pipeline.joblib")
metrics = {"r2_score": round(r2, 4), "mae": round(mae, 4), "rmse": round(rmse, 4), "cv_folds": CV_FOLDS,
           "cv_mean_r2": round(float(cv_scores.mean()), 4), "cv_std_r2": round(float(cv_scores.std()), 4),
           "cv_mean_mae": round(float(cv_mae.mean()), 4), "cv_std_mae": round(float(cv_mae.std()), 4),
           "cv_mean_rmse": round(float(cv_rmse.mean()), 4), "cv_std_rmse": round(float(cv_rmse.std()), 4),
           "train_size": len(X_train), "test_size": len(X_test), "total_records": len(df),
           "equation_terms": MODEL_TERMS, "intercept": float(reg.intercept_),
           "coefficients": coef, "centering": dict(zip(["distance_km", "engine_power_kw"], centers)),
           "raw_intercept": raw_b0, "raw_coefficients": raw_coef, "vif": vif, "diagnostics": diagnostics}
with open("metrics_colab.json", "w") as f:
    json.dump(metrics, f, indent=2)
try:
    from google.colab import files
    files.download("linear_regression_pipeline.joblib")
except ImportError:
    print("Saved linear_regression_pipeline.joblib and metrics_colab.json")

> The web application retrains this same model automatically whenever trip records or vehicle records change, so
> copying the `.joblib` file into the `model/` folder is optional. The notebook documents and reproduces the process.